# 01 — Read My Ears: 1:1 movement-detection replication

**Goal:** run their original movement-detection algorithm (Alves et al., CVPR Workshop 2025) directly on their own HuggingFace dataset [`joaomalves/read-my-ears`](https://huggingface.co/datasets/joaomalves/read-my-ears) — without modifying their code.

**Their algorithm pipeline:**
1. Per frame: YOLOv8l face + ear detection ([`jmalves5/horse-face-ear-detection`](https://github.com/jmalves5/horse-face-ear-detection))
2. Mask face region (black out background around the ear)
3. Crop ear region (face_width/2 × face_height/8, centered)
4. Optical flow (Farneback) on the ear-frame sequence; mean magnitude > threshold ⇒ movement

**Output:** precision/recall/F1/accuracy on the test split + comparison with the paper claim 87.5%.

**Requirements:** `bash setup.sh` completed (it pulls ultralytics, YOLOv8l weights, the HF dataset subset).


## 1. Sanity check — versions, weights, dataset

In [ ]:
import os, sys, glob
from pathlib import Path

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
os.chdir(POC)

import torch, cv2
from ultralytics import YOLO
import pandas as pd
import numpy as np

print('POC dir :', POC)
print('Python  :', sys.version.split()[0])
print('Torch   :', torch.__version__, '| MPS:', torch.backends.mps.is_available())

FACE_W = POC / 'vendor/horse-face-ear-detection/horse_face_detection/yolov8l_horse_face_detection.pt'
EAR_W = POC / 'vendor/horse-face-ear-detection/horse_ear_detection/yolov8l_horse_ear_detection.pt'
DATA = POC / 'vendor/ReadMyEars_Dataset/data'
for p in [FACE_W, EAR_W, DATA / 'test.csv', DATA / 'videos']:
    assert p.exists(), f'BRAK {p} — uruchom bash setup.sh'
    print(f'  ✓ {p.relative_to(POC)}')

# Load test.csv — expected format: video,label (label = 'action' or 'background')
test_df = pd.read_csv(DATA / 'test.csv')
print(f'\nTest split: {len(test_df)} clips; {(test_df.label=="action").sum()} action, {(test_df.label=="background").sum()} background')
available = test_df[test_df['video'].apply(lambda v: (DATA / v).exists())]
print(f'Locally available: {len(available)} clips (setup only pulls the S1 subset)')

## 2. Load their algorithm directly (import from `vendor/read-my-ears/movement-detection`)

In [ ]:
RME_MD = POC / 'vendor/read-my-ears/movement-detection'
if str(RME_MD) not in sys.path:
    sys.path.insert(0, str(RME_MD))

from locutils.ear_detection import get_ear_frames
from detect_movement import detect_movement
print('Original functions loaded:')
print(f'  get_ear_frames     <- {RME_MD}/locutils/ear_detection.py')
print(f'  detect_movement    <- {RME_MD}/detect_movement.py')

# Load YOLO weights
model_face = YOLO(str(FACE_W))
model_ear = YOLO(str(EAR_W))
print('\nYOLO ear classes :', model_ear.names)
print('YOLO face classes:', model_face.names)

## 3. Helper: face masking + frame extraction

Their original `EquinePainFaceDatasetMov` expects pre-processed `face_masked_clips/` from a separate folder. We do face masking on the fly: detect the face bbox, black out the background outside the mask.


In [ ]:
def read_frames_and_mask(clip_path, model_face, conf=0.5, target_size=(1920, 1080)):
    """Wczytaj klatki z mp4, ZRESIZUJ do (W, H) zgodnie z ich `transforms.Resize((1080, 1920))`,
    zaczernij wszystko poza face bbox.

    UWAGA: resize do 1080×1920 jest KLUCZOWY dla replikacji ich threshold=1 dla flow magnitudes.
    Without it flow_mag values come out ~3× smaller and the threshold has to be raised to ~3.0 (sweep).
    """
    cap = cv2.VideoCapture(str(clip_path))
    frames, masked_frames = [], []
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        # ICH preprocessing: upsample do FullHD przed YOLO + optical flow
        frame = cv2.resize(frame, target_size)
        frames.append(frame)
        # face mask: zachowaj tylko region z face bbox
        res = model_face.predict(frame, conf=conf, imgsz=320, verbose=False)
        boxes = res[0].boxes.cpu().numpy()
        masked = np.zeros_like(frame)
        if len(boxes.conf) > 0:
            i = int(np.argmax(boxes.conf))
            x1, y1, x2, y2 = map(int, boxes.xyxy[i])
            x1 = max(0, x1); y1 = max(0, y1)
            x2 = min(frame.shape[1], x2); y2 = min(frame.shape[0], y2)
            masked[y1:y2, x1:x2] = frame[y1:y2, x1:x2]
        else:
            masked = frame  # fallback: full frame
        masked_frames.append(masked)
    cap.release()
    return frames, masked_frames


## 4. Movement-detection pipeline per clip — calling their original functions

In [ ]:
from torchvision import transforms as tvt
import time

# Ich hardcoded constants z movement-detection/test.py
SKIP_FRAMES = 5
FLOW_MAG_THRESHOLD = 1
RESIZE = tvt.Resize((1080, 1920))

def predict_movement(clip_path, model_face, model_ear, skip=SKIP_FRAMES, threshold=FLOW_MAG_THRESHOLD):
    frames, masked = read_frames_and_mask(clip_path, model_face)
    if len(frames) < skip + 1:
        return None, 0.0, len(frames)
    ear_frames = get_ear_frames(
        filename=clip_path.name,
        frames=frames,
        masked_frames=masked,
        model_face=model_face,
        model_ear=model_ear,
    )
    if ear_frames.shape[0] == 0:
        return None, 0.0, len(frames)
    # detect_movement expects a list of tensors [(F, C, H, W)] — float [0,1]
    ear_normalized = ear_frames / 255.0
    output, mean_flow_mag = detect_movement(
        frames=[ear_normalized],
        skip_frames=skip,
        flow_mag_threshold=threshold,
    )
    pred = int(output[0].item())
    flow_mag = float(mean_flow_mag[0].item())
    return pred, flow_mag, len(frames)

## 5. Smoke test on 5 clips (mix of action + background)

In [ ]:
smoke = pd.concat([
    available[available['label'] == 'action'].head(3),
    available[available['label'] == 'background'].head(2),
])
print(f'Smoke test na {len(smoke)} klipach...')
rows = []
t0 = time.time()
for _, row in smoke.iterrows():
    clip = DATA / row['video']
    pred, flow_mag, n = predict_movement(clip, model_face, model_ear)
    gt = 1 if row['label'] == 'action' else 0
    rows.append({'clip': row['video'].split('/')[-1], 'gt': gt, 'pred': pred, 'flow_mag': round(flow_mag, 3), 'frames': n})
    print(f"  {row['video'].split('/')[-1]:35s} gt={gt} pred={pred} flow_mag={flow_mag:.3f} ({n} fr)")
print(f'\nCzas: {time.time()-t0:.1f}s')
smoke_df = pd.DataFrame(rows)
smoke_df

## 6. Full evaluation on the available test split

In [ ]:
# Inline kopia ich `calculate_metrics` z `vendor/read-my-ears/utils/metrics.py` —
# without wandb.log (which needs init) and without plt overhead in confusion_matrix.
def calculate_metrics(y_true, y_pred):
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / len(y_true) if y_true else 0
    cm = [[tn, fp], [fn, tp]]
    return precision, recall, f1, accuracy, cm

print(f'Full eval on {len(available)} test split clips (same as smoke + the remainder)...')
results = []
t0 = time.time()
for _, row in available.iterrows():
    clip = DATA / row['video']
    try:
        pred, flow_mag, n = predict_movement(clip, model_face, model_ear)
        if pred is None:
            pred = 0  # fallback for very short clips
        gt = 1 if row['label'] == 'action' else 0
        results.append({'clip': row['video'].split('/')[-1], 'gt': gt, 'pred': pred, 'flow_mag': round(flow_mag, 3), 'frames': n})
    except Exception as e:
        print(f"  BŁĄD na {row['video']}: {e}")
        continue
    if len(results) % 5 == 0:
        print(f'  {len(results)}/{len(available)} ({time.time()-t0:.1f}s)')
print(f'\nCompleted: {len(results)}/{len(available)} in {time.time()-t0:.1f}s')
res_df = pd.DataFrame(results)
res_df.to_csv(POC / 'outputs/rme_movement_predictions.csv', index=False)
print(f'Zapisano: outputs/rme_movement_predictions.csv')


## 7. Metrics (inlined `calculate_metrics` matching their code)

In [ ]:
y_gt = res_df['gt'].tolist()
y_pred = res_df['pred'].tolist()
precision, recall, f1, accuracy, cm = calculate_metrics(y_gt, y_pred)

metrics_summary = {
    'n_clips': len(res_df),
    'precision': round(precision, 4),
    'recall': round(recall, 4),
    'f1': round(f1, 4),
    'accuracy': round(accuracy, 4),
    'confusion_matrix': cm.tolist() if hasattr(cm, 'tolist') else cm,
    'paper_claim_accuracy': 0.875,
    'paper_delta': round(accuracy - 0.875, 4),
}
print('Movement Detection — replikacja Alves et al. 2025:')
print('-' * 50)
for k, v in metrics_summary.items():
    print(f'  {k:24s} {v}')

import json
with open(POC / 'outputs/movement_detection_results.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)
print(f'\nZapisano: outputs/movement_detection_results.json')

## 8. Confusion matrix plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cm_arr = np.array(cm) if not isinstance(cm, np.ndarray) else cm
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_arr, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['background', 'action']); ax.set_yticklabels(['background', 'action'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Ground truth')
ax.set_title(f'Read My Ears movement-detection\nN={len(res_df)}, accuracy={accuracy:.3f} (paper: 0.875)')
for i in range(cm_arr.shape[0]):
    for j in range(cm_arr.shape[1]):
        ax.text(j, i, str(cm_arr[i, j]), ha='center', va='center', fontsize=14, color='black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(POC / 'outputs/movement_detection_confusion_matrix.png', dpi=120)
plt.show()
print('Zapisano: outputs/movement_detection_confusion_matrix.png')

## 9. Decision note

**Item 3 from `../GATE.md`:**
- YES = movement-detection runs 1:1, accuracy near the paper claim (~0.875 ±10%)
- NO = run-time error OR accuracy <0.70 (paper not replicable on our subset)

**Technical notes:**
- We use YOLOv8**l** weights (~175 MB each) instead of yolov8n from their `configs.py` — `jmalves5/horse-face-ear-detection` only ships `l`
- Tested on the pulled S1 subset of the HF dataset (~20 clips) instead of the full test split (~80 clips) — accuracy tolerance ±10%
- Hardcoded thresholds: `SKIP_FRAMES=5`, `FLOW_MAG_THRESHOLD=1` (matching their `test.py`)
